# idd-figures layouts: `panel_grid` & `facet_grid`

A figure is a **tree of GridSpecs**. Every position is explicit: we never call `tight_layout` / `constrained_layout`, and saving uses `bbox_inches=None`. Composites are *config over these two layouts*, not bespoke drawing functions.

- `facet_grid(data, painter, row=, col=, sharex=, sharey=, ...)` — the common faceting cases.
- `panel_grid(spec, figsize=)` — explicit trees (nested grids, reserved label/legend/colorbar slots, mixed painters).

Import-only, made-up data.

In [ ]:
import matplotlib.pyplot as plt

from idd_figures.lib import example_data as ed
from idd_figures.lib import examples as ex
from idd_figures.lib.frames import panel_slice
from idd_figures.lib.layouts.grids import facet_grid
from idd_figures.lib.painters.lines import lines_panel

## facet_grid — small multiples over a group (shared y)
One shared count multiplier is computed once in the builder and passed to every panel, so the panels agree on units.

In [ ]:
ex.exemplar_forecast_superregion_facet()

It is just config. The same call, by hand — note the `FORECAST_LINES` config dict carrying the columns/palette/anchor, and `sharey=True` (the flag that distinguishes a shared-y grid from an independent one):

In [ ]:
pdat = ed.make_forecast_panel_df()
groups = [g for g in sorted(pdat["group"].unique()) if g != "Global"]
sub = panel_slice(pdat, {"measure": "mort", "metric": "rate", "group": groups})
fig = facet_grid(
    sub,
    lines_panel,
    col="group",
    ncol=3,
    sharey=True,
    panel_kwargs={**ex.FORECAST_LINES, "show_ci": False},
    titles=lambda info: info["group"],
    figsize=(13, 6),
)
fig

## explicit `panel_grid` — 'all SSPs + each SSP alone'
The top-left panel shows everything; the other three each show one scenario. Not a clean facet, so it is built from an explicit panel list (all four share x and y).

In [ ]:
ex.exemplar_forecast_all_plus_each()

## nested grid
`measure × group` outer, `count / rate` inner (nested GridSpec), reserved label gutters + a legend row, **one shared count multiplier** across every count subpanel, shared x within each count/rate pair — and **no `tight_layout`**.

In [ ]:
ex.exemplar_nested_grid()

## Figure anatomy — where is everything?

Because we never use `tight_layout`, you need to *see* where each piece lands. `show_anatomy(fig)` boxes the figure outer bounds, each panel, and that panel's x/y labels, title, and tick-label regions — so the **margins are the gaps** between the outer box and each panel box.

In [ ]:
from idd_figures.lib.layouts.anatomy import show_anatomy

df = ed.make_timeseries_df(n_series=2)
fig, ax = plt.subplots(figsize=(8, 5))
lines_panel(ax, df, x="year_id", value="value", hue="series", xlabel="year", ylabel="value")
ax.set_title("anatomy demo")
show_anatomy(fig)
fig

In [ ]:
# works on any figure, including a grid
fig = ex.exemplar_forecast_superregion_facet()
show_anatomy(fig)
fig

## Fonts, ticks, rounding (these are *layout / caller* features)

Not painter features — set them on the axes/figure, or globally via `rcParams`. Change every font at once with an `rc_context`:

In [ ]:
with plt.rc_context(
    {
        "font.size": 14,
        "axes.titlesize": 18,
        "axes.labelsize": 15,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
    }
):
    fig = ex.exemplar_forecast_superregion_facet()
fig

Tick **placement** via locators, and tick **rounding** via `style.ui_tick_formatter` (which reuses `numbers.smart_ui_format`, so tick labels share the same rounding/separator conventions as the rest of the figures):

In [ ]:
from matplotlib.ticker import MaxNLocator, MultipleLocator

from idd_figures.lib.style import ui_tick_formatter

fig, ax = plt.subplots(figsize=(8, 5))
lines_panel(
    ax,
    ed.make_timeseries_df(n_series=2),
    x="year_id",
    value="value",
    hue="series",
    xlabel="year",
    ylabel="value",
)
ax.xaxis.set_major_locator(MultipleLocator(10))  # tick placement
ax.yaxis.set_major_locator(MaxNLocator(4))
ax.yaxis.set_major_formatter(ui_tick_formatter())  # rounding: 3 sig figs, middle-dot decimal
fig

## saving
Neither painters nor layouts save — the caller does, via `idd_figures.lib.io.save_figure`, which always uses `bbox_inches=None` (never `'tight'`):

```python
from idd_figures.lib.io import save_figure
save_figure(fig, '/your/output/dir/figure', formats=['pdf', 'png'])
```